# Run Bash Agent with HuggingFace Model

This notebook runs the trained CLI agent using local HuggingFace model inference.

## Prerequisites

- Run `02_grpo_training.ipynb` first to train and save the model
- Model checkpoint should be at `outputs/grpo_langgraph_cli/merged_model`

## Features

- **Structured tool calling**: Uses JSON-based tool calls (not legacy code block parsing)
- **Human-in-the-loop**: All commands require user confirmation before execution
- **Security**: Command allowlist and injection protection
- **Conversation history**: Full context maintained across turns

## Step 1: Setup and Configuration

In [ ]:
import os
import sys
import json

# Add bash_agent to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), "bash_agent"))

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Import configuration from bash_agent
from config import Config

# Create config with notebook-specific settings
config = Config()

# Override model path if needed (uncomment to change)
# config.model_path = "outputs/grpo_langgraph_cli/merged_model"

print(f"Model path: {config.model_path}")
print(f"Root directory: {config.root_dir}")
print(f"Allowed commands: {config.allowed_commands}")

## Step 2: Import the Bash Tool

Import from `bash_agent/bash.py` - provides secure command execution with:
- Command allowlist
- Injection protection
- Working directory tracking

In [ ]:
# Import Bash tool from bash_agent
from bash_agent.bash import Bash

# Initialize the bash tool with security features
bash = Bash(config)
print(f"Bash tool initialized. Working directory: {bash.cwd}")

## Step 3: Import Message Handler and LLM Interface

Import from `bash_agent/helpers.py`

In [ ]:
# Import Messages and HuggingFaceLLM from bash_agent
from bash_agent.helpers import Messages, HuggingFaceLLM

print("Imported Messages and HuggingFaceLLM from bash_agent.helpers")

In [ ]:
# HuggingFaceLLM was imported above from bash_agent.helpers
# It provides:
#   - Local model inference with HuggingFace transformers
#   - JSON tool call parsing from model responses
#   - Conversion of structured commands to bash commands
print("HuggingFaceLLM ready (imported from bash_agent.helpers)")

## Step 4: Load the Trained Model

In [ ]:
import shutil
from pathlib import Path

# Need to copy modeling_nemotron_h.py from base model to merged model. 
# 1. Setup paths
cache_base = Path("/home/workbench/.cache/huggingface/hub/models--nvidia--NVIDIA-Nemotron-Nano-9B-v2")
destination_dir = Path("/project/code/4-agent-customization/outputs/grpo_langgraph_cli/merged_model")
file_name = "modeling_nemotron_h.py"

# 2. Dynamically find the correct snapshot hash
# The 'main' file in 'refs' contains the hash of the currently active snapshot
head_hash_file = cache_base / "refs" / "main"

if head_hash_file.exists():
    # Read the hash (e.g., "a4fb57...")
    commit_hash = head_hash_file.read_text().strip()
    source_path = cache_base / "snapshots" / commit_hash / file_name
else:
    # Fallback: If refs/main doesn't exist, grab the most recent directory in snapshots
    print("Warning: refs/main not found. searching snapshots directory...")
    snapshots_dir = cache_base / "snapshots"
    # Find all subdirectories
    snapshots = [d for d in snapshots_dir.iterdir() if d.is_dir()]
    if not snapshots:
        raise FileNotFoundError(f"No snapshots found in {snapshots_dir}")
    # Sort by modification time (newest first) to guess the right one
    newest_snapshot = sorted(snapshots, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    source_path = newest_snapshot / file_name

# 3. Execute Copy
if not source_path.exists():
    raise FileNotFoundError(f"Source file missing: {source_path}")

# Ensure destination exists
destination_dir.mkdir(parents=True, exist_ok=True)

# Copy preserving metadata
shutil.copy2(source_path, destination_dir / file_name)

print(f"✅ Success! Copied {file_name}")
print(f"From: {source_path}")
print(f"To:   {destination_dir}")

In [ ]:
# Verify full merged model exists by viewing contents
model_path = config.model_path
if not os.path.exists(model_path):
    print(f"ERROR: Model not found at {model_path}")
    print("Please run 02_grpo_training.ipynb first to train and save the model.")
else:
    print(f"Model found at: {model_path}")
    print(f"\nContents:")
    for f in os.listdir(model_path)[:10]:
        print(f"  - {f}")

### 📋 Exercise

The `HuggingFaceLLM` class wraps HuggingFace Transformers to provide the same interface as the NIM-based LLM from the base agent. It loads the trained checkpoint from `config.model_path` (which points to `outputs/grpo_langgraph_cli/merged_model`), handles tokenization, and parses structured JSON tool calls from the model's output.

Implement `llm` by instantiating `HuggingFaceLLM` with the `config` object.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
llm = HuggingFaceLLM(config)
```
</details>

In [ ]:
# TODO: Load the model with HuggingFaceLLM
llm = HuggingFaceLLM(config)

## Step 5: Test the Model

Quick test to verify the model generates proper JSON tool calls.

In [ ]:
# Test queries - use the JSON system prompt (matches training!)
# These prompts match the style of our training data
test_queries = [
    "Create a new project at ./assistant using the react-agent-python template",
    "Start the dev server on port 8080 without opening a browser",
    "Build a Docker image and tag it as myapp:v2",
]

# IMPORTANT: Use json_system_prompt - this is what the model was trained with!
print("Using json_system_prompt (matches training):")
print(config.json_system_prompt[:200] + "...")
print("=" * 60)

for query in test_queries:
    print(f"\n[Query] {query}")
    
    # Create fresh messages with the JSON system prompt (not generic bash prompt)
    test_messages = Messages(config.json_system_prompt)
    test_messages.add_user_message(query)
    
    response, tool_calls = llm.query(test_messages)
    
    # Clean up response for display
    display_response = response
    if "</think>" in display_response:
        display_response = display_response.split("</think>")[-1].strip()
    
    if tool_calls:
        for tc in tool_calls:
            args = json.loads(tc["function"]["arguments"])
            print(f"[Tool Call] {tc['function']['name']}: {args.get('cmd', args)}")
    else:
        print("[Tool Call] None detected")
    
    print("-" * 40)

## Step 6: Interactive Agent Loop

Run the full agent with human-in-the-loop confirmation. 

**Commands:**
- Type your request and press Enter
- When a command is proposed, type `y` to execute or `n` to decline
- Type `quit` or `exit` to stop

### 📋 Exercise

Implement `messages` by creating a `Messages` instance with `config.json_system_prompt`.

This is a subtle but critical detail: the model was trained with `config.json_system_prompt`, which instructs it to produce **structured JSON tool calls**. If you use the generic `config.system_prompt` instead, the model's output format won't match what it learned during GRPO training, and performance will degrade.

> 💡 **Why this matters**: During training, the system prompt was part of every input. The model learned to produce correct outputs *conditioned on that specific prompt*. Changing the prompt at inference time is like studying for one exam and sitting for a different one.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
messages = Messages(config.json_system_prompt)
```
</details>

### 📋 Exercise

Implement the execution block: if the user confirms the command, execute it with `bash.exec_bash_command(command)` and store the result in `tool_result`.

Even though the trained model is more accurate, the HITL pattern from `bash_agent.md` still applies. A smarter model reduces the frequency of errors but doesn't eliminate them—especially for edge cases outside the training distribution. The `confirm_execution()` function prompts the user before any command runs.

<details>
<summary>💡 NEED SOME HELP?</summary>

---

```python
if confirm_execution(command):
    tool_result = bash.exec_bash_command(command)
```
</details>

In [ ]:
def confirm_execution(cmd: str) -> bool:
    """Ask user to confirm command execution."""
    response = input(f"    Execute '{cmd}'? [y/N]: ").strip().lower()
    return response == "y" or response == "yes"


def run_agent_loop():
    """Main agent interaction loop."""
    
    # TODO: Initialize conversation with JSON system prompt (matches training)
    messages = Messages(config.json_system_prompt)
    
    print("\n" + "=" * 60)
    print("Bash Computer Use Agent (HuggingFace)")
    print("=" * 60)
    print(f"Model: {config.model_path}")
    print(f"Working directory: {bash.cwd}")
    print("Type 'quit' or 'exit' to stop.")
    print("Type 'clear' to reset conversation.")
    print("=" * 60 + "\n")
    
    while True:
        try:
            user_input = input(f"['{bash.cwd}'] > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\nShutting down. Bye!")
            break
        
        if user_input.lower() in ["quit", "exit"]:
            print("\nShutting down. Bye!")
            break
        
        if user_input.lower() == "clear":
            messages.clear()
            print("Conversation cleared.\n")
            continue
        
        if not user_input:
            continue
        
        # Add context about current directory
        user_with_context = f"{user_input}\nCurrent working directory: `{bash.cwd}`"
        messages.add_user_message(user_with_context)
        
        # Agent loop - may involve multiple tool calls
        while True:
            print("\nThinking...")
            
            try:
                response, tool_calls = llm.query(messages)
            except Exception as e:
                print(f"Error querying model: {e}")
                break
            
            # Clean response for display
            display_response = response.strip()
            if "</think>" in display_response:
                display_response = display_response.split("</think>")[-1].strip()
            
            if display_response:
                messages.add_assistant_message(display_response)
            
            # Process tool calls
            if tool_calls:
                for tc in tool_calls:
                    function_name = tc["function"]["name"]
                    function_args = json.loads(tc["function"]["arguments"])
                    tool_id = tc["id"]
                    
                    if function_name != "exec_bash_command" or "cmd" not in function_args:
                        tool_result = {"error": "Incorrect tool or function argument"}
                    else:
                        command = function_args["cmd"]
                        print(f"\nProposed command: {command}")
                        
                        # TODO: Execute the command after user confirmation
                        if confirm_execution(command):
                            tool_result = bash.exec_bash_command(command)
                            
                            if tool_result.get("stdout"):
                                print(f"\nOutput:\n{tool_result['stdout']}")
                            if tool_result.get("stderr"):
                                print(f"\nError:\n{tool_result['stderr']}")
                            if tool_result.get("error"):
                                print(f"\nError:\n{tool_result['error']}")
                        else:
                            tool_result = {"error": "The user declined to execute this command."}
                    
                    messages.add_tool_message(json.dumps(tool_result), tool_id)
            else:
                # No tool calls - show assistant message and break
                if display_response:
                    print(f"\n{display_response}")
                print("-" * 60)
                break

print("Agent loop defined. Run the next cell to start the interactive agent.")

In [ ]:
# Start the interactive agent
# Note: This cell requires interactive input. Stop with 'quit' or 'exit'
run_agent_loop()

## Summary

This notebook imports from the `bash_agent` module:

| Import | Source | Description |
|--------|--------|-------------|
| `Config` | `bash_agent/config.py` | Model path, security settings, system prompt |
| `Bash` | `bash_agent/bash.py` | Secure command execution with allowlist |
| `Messages` | `bash_agent/helpers.py` | Conversation history management |
| `HuggingFaceLLM` | `bash_agent/helpers.py` | Local model inference with JSON parsing |

### bash_agent Module Structure

```
bash_agent/
├── config.py      # Configuration class
├── bash.py        # Bash tool with security features
├── helpers.py     # Messages and HuggingFaceLLM classes
├── prompts.py     # System prompts (JSON-based, no legacy)
└── main_hf.py     # CLI entry point for the agent
```

### Key Features

- **Structured tool calling**: JSON-based tool calls (not code block parsing)
- **Human-in-the-loop**: All commands require user confirmation
- **Security**: Command allowlist and injection protection
- **Conversation history**: Full context maintained across turns

### Next Steps

- Train longer for better accuracy (increase `max_steps` in training)
- Add more commands to the allowlist in `bash_agent/config.py`
- Run from CLI: `python bash_agent/main_hf.py`
- Deploy as a server with vLLM for production use